# NITS-IQA analysis (English publication adaptation)

This notebook is adapted from a historical experiment notebook for an English-only repository. The original notebook remains in the local archive. Source data are the 405 recorded pairs; no source images or model inference are included. Numeric calculations retain the historical logic, with an equivalent binary balanced-accuracy implementation and explicit repairs of invalid or out-of-order cells.

In [ ]:
from pathlib import Path
import pandas as pd

def locate_nits_csv():
    """Find the English-header NITS-IQA scores relative to the notebook or repository."""
    base = Path.cwd().resolve()
    candidates = [base / "nits_iqa/verified/NITS_IQA_405_verified.csv",
                  base / "../verified/NITS_IQA_405_verified.csv",
                  base / "verified/NITS_IQA_405_verified.csv"]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError("NITS_IQA_405_verified.csv not found; run from repository root or notebook directory")

DATA_CSV = locate_nits_csv()

def fast_balanced_accuracy(y_true, y_pred):
    """Binary balanced accuracy, equivalent to sklearn on these two-class folds."""
    import numpy as np
    yt = np.asarray(y_true)
    yp = np.asarray(y_pred)
    pos = yt == 1
    neg = yt == 0
    if not pos.any() or not neg.any():
        raise ValueError("Both classes are required for this binary benchmark")
    return 0.5 * ((yp[pos] == 1).mean() + (yp[neg] == 0).mean())


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

file = DATA_CSV
df = pd.read_csv(file)

metrics = [
    "DINOv2-Base (%)",
    "DINOv2-Large (%)",
    "Dists (%)",
    "SSIM (%)",
    "OpenCLIP (%)"
]

X = df[metrics].to_numpy(dtype=float)
y = (df["MOS"] >= 50).astype(int).to_numpy()
groups = df["Ref"].to_numpy()

methods = [
    "DISTS",
    "AND-5",
    "OR-5",
    "OpenCLIP + DISTS",
    "Logistic Fusion"
]

predictions = {
    method: np.zeros(len(df), dtype=int)
    for method in methods
}

logo = LeaveOneGroupOut()

for train_idx, test_idx in logo.split(X, y, groups):

    X_train = X[train_idx]
    X_test = X[test_idx]
    y_train = y[train_idx]

    thresholds = []

    # Calibrate each metric threshold on the training fold only
    for j in range(len(metrics)):

        candidates = np.unique(X_train[:, j])

        best_threshold = candidates[0]
        best_score = -1

        for threshold in candidates:

            pred = (X_train[:, j] >= threshold).astype(int)

            score = fast_balanced_accuracy(y_train, pred)

            if score > best_score:
                best_score = score
                best_threshold = threshold

        thresholds.append(best_threshold)

    thresholds = np.array(thresholds)

    passed = X_test >= thresholds

    predictions["DISTS"][test_idx] = passed[:, 2].astype(int)

    predictions["AND-5"][test_idx] = (
        passed.all(axis=1).astype(int)
    )

    predictions["OR-5"][test_idx] = (
        passed.any(axis=1).astype(int)
    )

    predictions["OpenCLIP + DISTS"][test_idx] = (
        passed[:, 2] & passed[:, 4]
    ).astype(int)

    # Learned fusion
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000)
    )

    model.fit(X_train, y_train)

    predictions["Logistic Fusion"][test_idx] = (
        model.predict(X_test)
    )

results = []

for method in methods:

    pred = predictions[method]

    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    results.append({
        "Method": method,
        "Accuracy": accuracy_score(y, pred),
        "Balanced_Accuracy": fast_balanced_accuracy(y, pred),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "FPR": fp / (fp + tn),
        "FNR": fn / (fn + tp),
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn
    })

comparison_df = pd.DataFrame(results)

print(comparison_df.round(4).to_string(index=False))

output_dir = Path("results")
output_dir.mkdir(parents=True, exist_ok=True)

comparison_df.to_csv(
    output_dir / "oracle_comparison.csv",
    index=False
)

In [ ]:
import pandas as pd
from pathlib import Path

# Extract distortion type and severity level from the filename
df["Distortion"] = df["Distorted"].str.extract(r"D(\d+)L\d+")[0]
df["Level"] = df["Distorted"].str.extract(r"D\d+L(\d+)")[0]

# Check that distorted filenames follow the expected convention
print("Unparsed filenames:", df["Distortion"].isna().sum())
print("Distortion codes:", sorted(df["Distortion"].dropna().unique()))
print("Severity levels:", sorted(df["Level"].dropna().unique()))

records = []

for method, pred in predictions.items():

    temp = df.copy()
    temp["Actual"] = y
    temp["Predicted"] = pred

    for distortion, group in temp.groupby("Distortion"):

        actual = group["Actual"]
        predicted = group["Predicted"]

        fp = ((actual == 0) & (predicted == 1)).sum()
        fn = ((actual == 1) & (predicted == 0)).sum()

        records.append({
            "Method": method,
            "Distortion": distortion,
            "Samples": len(group),
            "False_Accepts": fp,
            "False_Rejects": fn,
            "Errors": fp + fn
        })

error_df = pd.DataFrame(records)

print("\nERRORS BY DISTORTION")
print(error_df.to_string(index=False))

output_dir = Path("results")
output_dir.mkdir(parents=True, exist_ok=True)

error_df.to_csv(
    output_dir / "errors_by_distortion.csv",
    index=False
)

In [ ]:
import pandas as pd
from pathlib import Path

# Uses df, y, and predictions from the preceding cells
df["Distortion"] = df["Distorted"].str.extract(r"D(\d+)L\d+")[0]
df["Level"] = df["Distorted"].str.extract(r"D\d+L(\d+)")[0]

df["Actual"] = y
df["DISTS_Prediction"] = predictions["DISTS"]

# Actual class distribution for each distortion type
summary = []

for distortion, group in df.groupby("Distortion"):

    actual = group["Actual"]
    pred = group["DISTS_Prediction"]

    tn = ((actual == 0) & (pred == 0)).sum()
    fp = ((actual == 0) & (pred == 1)).sum()
    fn = ((actual == 1) & (pred == 0)).sum()
    tp = ((actual == 1) & (pred == 1)).sum()

    summary.append({
        "Distortion": distortion,
        "Acceptable": int(tp + fn),
        "Unacceptable": int(tn + fp),
        "DISTS_FPR": round(fp / (tn + fp), 4)
            if (tn + fp) > 0 else None,
        "DISTS_FNR": round(fn / (tp + fn), 4)
            if (tp + fn) > 0 else None,
        "Mean_MOS": round(group["MOS"].mean(), 2)
    })

summary_df = pd.DataFrame(summary)

print("DISTORTION ANALYSIS")
print(summary_df.to_string(index=False))

# Misclassified samples for distortion codes D4 and D5
errors = df[
    (df["Distortion"].isin(["4", "5"])) &
    (df["Actual"] != df["DISTS_Prediction"])
].copy()

columns = [
    "Ref",
    "Distorted",
    "Distortion",
    "Level",
    "MOS",
    "Dists (%)",
    "OpenCLIP (%)",
    "SSIM (%)",
    "Actual",
    "DISTS_Prediction"
]

print("\nD4 AND D5 MISCLASSIFIED IMAGES")
print(errors[columns].to_string(index=False))

output_dir = Path("results")
output_dir.mkdir(parents=True, exist_ok=True)

summary_df.to_csv(
    output_dir / "distortion_analysis.csv",
    index=False
)

errors[columns].to_csv(
    output_dir / "d4_d5_errors.csv",
    index=False
)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix
)

# 1. Load the data
file = DATA_CSV

df = pd.read_csv(file)

metrics = [
    "DINOv2-Base (%)",
    "DINOv2-Large (%)",
    "Dists (%)",
    "SSIM (%)",
    "OpenCLIP (%)"
]

X = df[metrics].to_numpy(dtype=float)
y = (df["MOS"] >= 50).astype(int).to_numpy()
groups = df["Ref"].to_numpy()

# 2. Basic dataset checks
assert len(df) == 405
assert len(np.unique(groups)) == 9
assert y.sum() == 199

print("PASS: Dataset validation")

# 3. Recompute out-of-fold predictions for every metric
oof = np.zeros((len(df), len(metrics)), dtype=int)

logo = LeaveOneGroupOut()

for train_idx, test_idx in logo.split(X, y, groups):

    # Check that reference groups do not leak across folds
    assert set(groups[train_idx]).isdisjoint(
        set(groups[test_idx])
    )

    for j in range(len(metrics)):

        candidates = np.unique(X[train_idx, j])

        best_threshold = candidates[0]
        best_score = -1

        for threshold in candidates:

            train_pred = (
                X[train_idx, j] >= threshold
            ).astype(int)

            score = fast_balanced_accuracy(
                y[train_idx], train_pred
            )

            if score > best_score:
                best_score = score
                best_threshold = threshold

        oof[test_idx, j] = (
            X[test_idx, j] >= best_threshold
        ).astype(int)

print("PASS: Cross-validation completed")

# 4. DISTS confusion matrix
tn, fp, fn, tp = confusion_matrix(
    y, oof[:, 2], labels=[0, 1]
).ravel()

print("\nDISTS RESULTS")
print("TP:", tp)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)

assert (tp, tn, fp, fn) == (162, 170, 36, 37)

print("PASS: DISTS confusion matrix")

# 5. Error complementarity
dists_errors = oof[:, 2] != y

expected = {
    0: 15,
    1: 15,
    3: 19,
    4: 23
}

print("\nERROR COMPLEMENTARITY")

for j, expected_count in expected.items():

    corrected = int(
        np.sum(dists_errors & (oof[:, j] == y))
    )

    print(metrics[j], "corrected:", corrected)

    assert corrected == expected_count

print("\nALL VALIDATION CHECKS PASSED")

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import LeaveOneGroupOut

# 1. Dataset validation
assert len(df) == 405
assert len(np.unique(groups)) == 9
assert int(y.sum()) == 199
assert int((y == 0).sum()) == 206

print("PASS: Dataset validation")

# 2. Check reference-group leakage in cross-validation
logo = LeaveOneGroupOut()

for train_idx, test_idx in logo.split(X, y, groups):
    train_refs = set(groups[train_idx])
    test_refs = set(groups[test_idx])

    assert train_refs.isdisjoint(test_refs)

print("PASS: Reference group separation")

# 3. Independently validate DISTS predictions
tn, fp, fn, tp = confusion_matrix(y, oof[:, 2], labels=[0, 1]).ravel()

assert (tp, tn, fp, fn) == (162, 170, 36, 37)

print("PASS: DISTS confusion matrix")

# 4. Compare with the earlier DISTS predictions
assert np.array_equal(
    oof[:, 2],
    predictions["DISTS"]
)

print("PASS: Independent prediction consistency")

# 5. Check error complementarity
expected = {
    0: 15,
    1: 15,
    3: 19,
    4: 23
}

dists_errors = oof[:, 2] != y

for j, expected_count in expected.items():
    corrected = int(
        np.sum(dists_errors & (oof[:, j] == y))
    )

    assert corrected == expected_count

print("PASS: Error complementarity")

print("\nALL VALIDATION CHECKS PASSED")